# Fetch Test Data

Fetch OpenSky trajectories for one airport and time period, apply the project preprocessing, retain departures, compute cumulative distance, and save the resulting dataset.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from traffic.core import Traffic
from traffic.data import airports, opensky

## Configuration

In [2]:
AIRPORT = "EBBR"
START = "2025-11-05 00:00"
STOP = "2025-11-06 00:00"
SEASON = "Autumn"

MAX_WORKERS = 1

valid_seasons = {"Spring", "Summer", "Autumn", "Winter"}
if SEASON not in valid_seasons:
    raise ValueError(
        f"Invalid season '{SEASON}'. Expected one of: {sorted(valid_seasons)}"
    )

# Project and data directories
PROJECT_ROOT = Path.home() / "Projects" / "OSN_pushback_detection"
DATA_ROOT = PROJECT_ROOT / "data"

test_root = DATA_ROOT / "test" / "01_fetched"

OUTPUT_DIR = test_root / AIRPORT / SEASON
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = f"{pd.Timestamp(START):%d-%m-%y}.parquet"
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FILE

bounds = airports[AIRPORT].shape.convex_hull.buffer(0.1)

OUTPUT_PATH

PosixPath('/cluster/home/hanv/Projects/OSN_pushback_detection/data/test/01_fetched/EBBR/Autumn/05-11-25.parquet')

## Fetch OpenSky Data

In [3]:
timestamps = pd.date_range(
    START,
    STOP,
    freq="24h",
)

data = []

for t1, t2 in zip(
    timestamps[:-1],
    timestamps[1:],
):
    print(t1, t2)

    flight_list = opensky.flightlist(
        start=t1,
        stop=t2,
        airport=AIRPORT,
    )

    if flight_list is None:
        continue

    icao24 = flight_list.icao24

    tmp = opensky.history(
        start=str(t1),
        stop=str(t2),
        bounds=bounds,
        icao24=icao24,
        return_flight=True,
    )

    if tmp is not None:
        data.append(tmp.data)

if not data:
    raise RuntimeError("No OpenSky data returned for the selected period.")

trajs = Traffic(
    pd.concat(
        data,
        ignore_index=True,
    )
)

trajs

2025-11-05 00:00:00 2025-11-06 00:00:00


,,count
icao24,callsign,
44cdce,BEL8KG,3766
4b17e2,SWR791T,3303
4ca41e,IRL283,3115
44ce72,BEL9AN,3017
44cc42,BEL1844,2996
44ccd8,BEL3DY,2909
44cc42,BEL5FT,2886
4b17e1,SWR9DV,2558
440713,EJU69HX,2451


## Preprocessing

In [4]:
def convert_timestamp(value):
    if isinstance(value, (float, int)) and value > 1e9:
        return pd.to_datetime(
            value,
            unit="s",
            errors="coerce",
        )

    if isinstance(value, pd.Timestamp):
        return value

    return pd.NaT


trajs.data["timestamp"] = trajs.data["timestamp"].apply(convert_timestamp)
trajs.data["hour"] = trajs.data["hour"].apply(convert_timestamp)

trajs.data["onground"] = trajs.data["onground"].astype(np.bool_)
trajs.data["alert"] = trajs.data["alert"].astype(np.bool_)
trajs.data["spi"] = trajs.data["spi"].astype(np.bool_)

trajs.data["last_position"] = trajs.data["last_position"].astype(np.float64)

trajs.data["latitude"] = trajs.data["latitude"].astype("double")
trajs.data["longitude"] = trajs.data["longitude"].astype("double")
trajs.data["groundspeed"] = trajs.data["groundspeed"].astype("double")
trajs.data["track"] = trajs.data["track"].astype("double")
trajs.data["vertical_rate"] = trajs.data["vertical_rate"].astype("double")
trajs.data["altitude"] = trajs.data["altitude"].astype("double")
trajs.data["geoaltitude"] = trajs.data["geoaltitude"].astype("double")
trajs.data["lastcontact"] = trajs.data["lastcontact"].astype("double")

trajs.data["icao24"] = trajs.data["icao24"].astype("object")
trajs.data["callsign"] = trajs.data["callsign"].astype("object")
trajs.data["squawk"] = trajs.data["squawk"].astype("object")

trajs = trajs.assign_id().eval(
    max_workers=1,
    desc="Assigning flight IDs",
)

trajs = trajs.resample("1s").eval(
    max_workers=MAX_WORKERS,
    desc="Resampling",
)

trajs

Assigning flight IDs:   0%|          | 0/488 [00:00<?, ?it/s]

Resampling:   0%|          | 0/488 [00:00<?, ?it/s]

,count
flight_id,
BEL8KG_197,4325
SWR791T_372,3371
BEL9AN_253,3145
BEL5FT_126,2886
BEL1844_125,2630
SWR9DV_370,2595
EJU69HX_111,2464
IRL283_396,2138
BEL3DY_164,2029


## Retain Departures

In [5]:
def dep_runway(flight):
    takeoff = flight.takeoff(
        AIRPORT,
        method="track_based",
        adv_filtering=True,
    )

    if takeoff is not None:
        next_segment = takeoff.next()

        if next_segment is not None:
            runway = next_segment.max("runway")

            return {
                "Flight ID": flight.flight_id,
                "Runway": runway,
            }


departure_runway = (
    trajs
    .iterate_lazy()
    .pipe(dep_runway)
    .eval(
        max_workers=MAX_WORKERS,
        desc="Determining departure flights",
    )
)

if departure_runway is None:
    raise RuntimeError("No departure flights detected.")

deps = trajs[
    departure_runway["Flight ID"].to_list()
]

deps

Determining departure flights:   0%|          | 0/488 [00:00<?, ?it/s]

/cluster/home/hanv/Projects/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/cluster/home/hanv/Projects/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/cluster/home/hanv/Projects/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/cluster/home/hanv/Projects/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/cluster/home/hanv/Projects/.venv/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/cluster/home/hanv/Projects/.v

,count
flight_id,
SWR791T_372,3371
BEL9AN_253,3145
BEL5FT_126,2886
SWR9DV_370,2595
EJU69HX_111,2464
IRL283_396,2138
CAZ301_373,2016
AIH588_452,1833
BCS3UP_065,1691


## Compute Cumulative Distance

In [6]:
deps = deps.cumulative_distance().eval(
    max_workers=MAX_WORKERS,
    desc="Computing cumulative distance",
)

deps

Computing cumulative distance:   0%|          | 0/223 [00:00<?, ?it/s]

,count
flight_id,
SWR791T_372,3371
BEL9AN_253,3145
BEL5FT_126,2886
SWR9DV_370,2595
EJU69HX_111,2464
IRL283_396,2138
CAZ301_373,2016
AIH588_452,1833
BCS3UP_065,1691


## Save Dataset

In [7]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

deps.to_parquet(OUTPUT_PATH)

print(f"Saved dataset: {OUTPUT_PATH}")

Saved dataset: /cluster/home/hanv/Projects/OSN_pushback_detection/data/test/01_fetched/EBBR/Autumn/05-11-25.parquet


## Quick Check

In [8]:
deps.sample(10).scatter_map(
    zoom=10,
    hover_data=["flight_id", "timestamp"],
)